# Data Download and Prepare

This Notebook will download all the data used for this project. We will split the data into training and evaluation. So we can fine-tune the pretrained model and evaluate all changes in this research. 

## Data Sources

Since part of this and future research concentrate on energy forecast the energy domain is selected as our main focus. So, most data comes from the energy domain but some data is also selected from other areas like the M6, M5 and M4 competition datasets to get some benchmarks in other areas.

- Home Electricity, 15min, 100k timestamps, Home meter data from a single household
- OPSD Time Series (Europe), https://doi.org/10.25832/time_series/2019-06-05
- Renewable Energy and Electricity Demand (US), https://openenergyhub.ornl.gov/explore/dataset/renewable-energy-and-electricity-demand-time-series-dataset-with-exogenous-varia/information/?utm_source=chatgpt.com
- PJM Hourly Energy Consumption (US), https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption
- M6, M5, M4 Competition Datasets, https://forecasters.org/resources/time-series-data/

In [18]:
import polars as pl
import numpy as np
import niquests
from io import BytesIO
from pathlib import Path
from tqdm import tqdm

In [5]:
main_data_folder = Path("../data")
main_data_folder.mkdir(exist_ok=True)

In [111]:
def download_data(url: str, filepath: Path) -> None:
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with niquests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(filepath, "wb") as f, tqdm(
            desc=filepath.name,
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))


def split_data(
    df: pl.DataFrame,
    time_col: str = "timestamp",
    train_ratio: float = 0.8,
    group_cols: list[str] | None = None,
    index_col: str = "series_index",
    index_start: int = 0,
) -> tuple[pl.DataFrame, pl.DataFrame, int]:
    
    if group_cols is None:
        df = df.with_columns(pl.lit(index_start).alias(index_col)).sort(time_col)
        n = len(df)
        train_df = df[: int(n * train_ratio)]
        test_df = df[int(n * train_ratio) :]
        max_group_index = df.select(pl.col(index_col).max()).item()
        return train_df, test_df, max_group_index

    # ensure sorted within each group
    df = df.sort(group_cols + [time_col])
    # add per-group index and group length
    df = df.with_columns([
        (pl.struct(group_cols)
         .rank("dense")
         .cast(pl.Int64) - 1 + index_start).alias(index_col),
        pl.int_range(0, pl.len()).over(group_cols).alias("group_index"),
        pl.len().over(group_cols).alias("group_size"),
    ])
    max_group_index = df.select(pl.col(index_col).max()).item()
    # compute split point (last X% = test)
    df = df.with_columns(
        (pl.col("group_size") * train_ratio).floor().cast(pl.Int64).alias("split_idx")
    )
    # split
    train_df = df.filter(pl.col("group_index") < pl.col("split_idx")).drop(["group_index", "group_size", "split_idx"])
    test_df  = df.filter(pl.col("group_index") >= pl.col("split_idx")).drop(["group_index", "group_size", "split_idx"])

    return train_df, test_df, max_group_index

# Home Electricity

This is a personal dataset from a single household with 80m².

In [ ]:
# download data
data_url_home = "https://raw.githubusercontent.com/Richi0D/tirex_loss/refs/heads/main/data/home_electricity/home_electricity.csv"
data_path_home = main_data_folder / "home_electricity" / "home_electricity.csv"
download_data(data_url_home, data_path_home)

home_electricity.csv: 3.63MB [00:00, 56.6MB/s]                  


In [ ]:
# split data
df_home_electricity = pl.read_csv(data_path_home, try_parse_dates=True)
df_home_electricity = df_home_electricity.rename({"Datum": "timestamp", "kW": "value"})
df_home_electricity = df_home_electricity.with_columns(pl.lit("home_electricity").alias("series_name"))
df_home_electricity_train, df_home_electricity_test, max_group_index = split_data(df_home_electricity)
print(f"Columns: {df_home_electricity_train.columns}")
print(f"Train Shape: {df_home_electricity_train.shape[0]}")
print(f"Test Shape: {df_home_electricity_test.shape[0]}")
df_home_electricity_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 80640
Test Shape: 20160


timestamp,value,series_name,series_index
"datetime[μs, UTC]",f64,str,i32
2023-06-10 22:00:00 UTC,0.308,"""home_electricity""",0
2023-06-10 22:15:00 UTC,0.14,"""home_electricity""",0
2023-06-10 22:30:00 UTC,0.14,"""home_electricity""",0
2023-06-10 22:45:00 UTC,0.14,"""home_electricity""",0
2023-06-10 23:00:00 UTC,0.148,"""home_electricity""",0


# OPSD

Load, wind, solar and prices from 32 European countries.
15min and 60min stacked dataset is used.

In [ ]:
# download data
data_url_opsd_15m = "https://data.open-power-system-data.org/time_series/latest/time_series_15min_stacked.csv"
data_url_opsd_60m = "https://data.open-power-system-data.org/time_series/latest/time_series_60min_stacked.csv"
data_path_opsd_15m = main_data_folder / "opsd" / "opsd_15min_stacked.csv"
data_path_opsd_60m = main_data_folder / "opsd" / "opsd_60min_stacked.csv"
download_data(data_url_opsd_15m, data_path_opsd_15m)
download_data(data_url_opsd_60m, data_path_opsd_60m)

opsd_15min_stacked.csv: 669MB [00:09, 72.4MB/s]
opsd_60min_stacked.csv: 841MB [00:13, 65.1MB/s]


In [ ]:
# split data
df_opsd_15m = pl.read_csv(data_path_opsd_15m, try_parse_dates=True)
df_opsd_15m = df_opsd_15m.rename({"utc_timestamp": "timestamp", "data": "value"})
df_opsd_15m = df_opsd_15m.with_columns(
    pl.concat_str([pl.col("region"), pl.col("variable"), pl.col("attribute")], separator="_").alias("series_name")
).drop(["region", "variable", "attribute"])
df_opsd_15m_train, df_opsd_15m_test, max_group_index = split_data(df_opsd_15m,
                                                group_cols=["series_name"],
                                                index_start=max_group_index + 1)
print(f"Columns: {df_opsd_15m_train.columns}")
print(f"Train Shape: {df_opsd_15m_train.shape[0]}")
print(f"Test Shape: {df_opsd_15m_test.shape[0]}")
df_opsd_15m_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 8479014
Test Shape: 2119784


timestamp,value,series_name,series_index
"datetime[μs, UTC]",f64,str,i64
2015-01-01 00:15:00 UTC,5966.8,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 00:30:00 UTC,5935.6,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 00:45:00 UTC,5934.4,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 01:00:00 UTC,5750.8,"""AT_load_actual_entsoe_transpar…",1
2015-01-01 01:15:00 UTC,5777.6,"""AT_load_actual_entsoe_transpar…",1


In [ ]:
# split data
df_opsd_60m = pl.read_csv(data_path_opsd_60m, try_parse_dates=True)
df_opsd_60m = df_opsd_60m.rename({"utc_timestamp": "timestamp", "data": "value"})
df_opsd_60m = df_opsd_60m.with_columns(
    pl.concat_str([pl.col("region"), pl.col("variable"), pl.col("attribute")], separator="_").alias("series_name")
).drop(["region", "variable", "attribute"])
df_opsd_60m_train, df_opsd_60m_test, max_group_index = split_data(df_opsd_60m,
                                                group_cols=["series_name"],
                                                index_start=max_group_index + 1)
print(f"Columns: {df_opsd_60m_train.columns}")
print(f"Train Shape: {df_opsd_60m_train.shape[0]}")
print(f"Test Shape: {df_opsd_60m_test.shape[0]}")
df_opsd_60m_train.head()

Columns: ['timestamp', 'value', 'series_name', 'series_index']
Train Shape: 10949008
Test Shape: 2737395


timestamp,value,series_name,series_index
"datetime[μs, UTC]",f64,str,i64
2015-01-01 00:00:00 UTC,5946.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 01:00:00 UTC,5726.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 02:00:00 UTC,5347.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 03:00:00 UTC,5249.0,"""AT_load_actual_entsoe_transpar…",60
2015-01-01 04:00:00 UTC,5309.0,"""AT_load_actual_entsoe_transpar…",60


# Open Energy Hub

Renewable Energy and Electricity Demand Time Series Dataset with Exogenous Variables at 5-minute Interval.

The database contains 12 columns:
- date
- station (1: Winter, 2: Spring, 3: Summer, 4: Autumn)
- day of the week (0: Monday, ... , 6: Sunday)
- DHI (W/m2)
- DNI (W/m2)
- GHI (W/m2)
- wind speed (m/s)
- humidity (%)
- temperature (degrees)
- solar energy production (MW)
- wind energy production (MW)
- electricity demand (MW)

We use only the last 3 columns which provide data about production and demand.

In [115]:
# download data
data_url_caiso = "https://data.mendeley.com/public-files/datasets/fdfftr3tc2/files/fff037a3-d0e4-496f-92f7-5c5820a734f1/file_downloaded"
data_path_caiso = main_data_folder / "openenergyhub" / "openenergyhub.csv"
download_data(data_url_caiso, data_path_caiso)

openenergyhub.csv: 100%|██████████| 28.4M/28.4M [00:05<00:00, 5.26MB/s]


In [116]:
# split data
df_caiso = pl.read_csv(data_path_caiso, try_parse_dates=True)
df_caiso = df_caiso.rename({"Time": "timestamp",
                            "PV_production": "caiso_pv_production",
                            "Wind_production": "caiso_wind_production",
                            "Electric_demand": "caiso_electric_demand"})
df_caiso = df_caiso.with_columns(
    pl.col("timestamp").str.strptime(pl.Datetime, format="%Y-%m-%d-T%H:%M", strict=False)
    )
df_caiso_melt = df_caiso.unpivot(index="timestamp",
                                on=["caiso_pv_production", "caiso_wind_production", "caiso_electric_demand"],
                                variable_name="series_name",
                                value_name="value")

df_caiso_train, df_caiso_test, max_group_index = split_data(df_caiso_melt,
                                                             group_cols=["series_name"],
                                                               index_start=max_group_index + 1)
print(f"Columns: {df_caiso_train.columns}")
print(f"Train Shape: {df_caiso_train.shape[0]}")
print(f"Test Shape: {df_caiso_test.shape[0]}")
df_caiso_train.head()

Columns: ['timestamp', 'series_name', 'value', 'series_index']
Train Shape: 757554
Test Shape: 189390


timestamp,series_name,value,series_index
datetime[μs],str,i64,i64
2019-01-01 00:00:00,"""caiso_electric_demand""",22216,358
2019-01-01 00:05:00,"""caiso_electric_demand""",22106,358
2019-01-01 00:10:00,"""caiso_electric_demand""",22130,358
2019-01-01 00:15:00,"""caiso_electric_demand""",22040,358
2019-01-01 00:20:00,"""caiso_electric_demand""",21963,358
